# Colab: Fine-tune YOLOv8n-pose under mixed surveillance corruptions

Upload `data/processed/colab_yolo_pose.zip` (from `scripts/09_package_colab_dataset.py`) to Drive, then run this notebook on a GPU runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!nvidia-smi

In [ ]:
%pip install -q ultralytics opencv-python-headless pycocotools PyYAML pandas matplotlib tqdm

In [ ]:
import zipfile
from pathlib import Path

ZIP = Path('/content/drive/MyDrive/colab_yolo_pose.zip')  # adjust path
OUT = Path('/content/robust_human')
OUT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP, 'r') as zf:
    zf.extractall(OUT)
print('Extracted to', OUT)
list((OUT/'data'/'yolo_pose').iterdir())

In [ ]:
from ultralytics import YOLO
data_yaml = OUT/'data'/'yolo_pose'/'dataset.yaml'
assert data_yaml.exists(), data_yaml
model = YOLO('yolov8n-pose.pt')
results = model.train(
    data=str(data_yaml),
    epochs=20,
    imgsz=416,
    patience=5,
    device=0,
    project=str(OUT/'runs'),
    name='pose_robust_ft',
    exist_ok=True,
)
best = Path(results.save_dir)/'weights'/'best.pt'
print('BEST:', best)

In [ ]:
import shutil
drive_out = Path('/content/drive/MyDrive/pose_robust_ft_best.pt')
shutil.copy2(best, drive_out)
print('Copied to', drive_out)
shutil.make_archive('/content/drive/MyDrive/pose_robust_ft_run', 'zip', results.save_dir)
print('Zipped training run to Drive')

## Optional: quick val check
Download `best.pt` locally and run `scripts/05_eval_finetuned.py`.